# 🦷 Phase 2: timm Dental Lesion Classifier Training & Evaluation

This notebook trains and evaluates the `DentalClassifier` (Stage 2) on the **Caries-Spectra** dataset to classify dental caries lesions across three severity stages:
- **Class 0**: `healthy` (NoEnamel_Caries)
- **Class 1**: `caries_early` (EarlyStageEnamel_Caries)
- **Class 2**: `caries_advanced` (AdvanceEnamel_Caries)

### Key Architectural Highlights
- **Backbone**: `timm` transfer learning backbone (`efficientnet_b0` by default, configurable to `convnext_tiny`).
- **Loss Function**: Class-weighted Cross-Entropy Loss to address medical class imbalance.
- **Stratified Split**: 70% Train (1,400 images) / 15% Val (300 images) / 15% Test (300 images).
- **Augmentations**: Albumentations pipeline (RandomResizedCrop, HorizontalFlip, Rotate, ColorJitter, ImageNet normalization).
- **Execution Strategy**: The training loop and dataset wrappers are encapsulated in `src/dental_model/classifier/train.py`, keeping this notebook clean, modular, and reproducible.

In [ ]:
# 1. Set Working Directory & Verify GPU
%matplotlib inline
import os
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

# Ensure current working directory is the repo root
if Path.cwd().name == "notebooks":
    os.chdir("..")

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"Working Directory : {REPO_ROOT}")
print(f"PyTorch Version   : {torch.__version__}")
print(f"CUDA Available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name   : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Total VRAM        : {props.total_memory / 1e9:.2f} GB")

## 2. 📊 Dataset & Class Distribution Verification

Load `data/processed/classifier/labels.csv` and verify the stratified 70/15/15 split across the 3 classes.

Expected distribution per the synopsis:
- **healthy**: 280 train / 60 val / 60 test (400 total)
- **caries_early**: 560 train / 120 val / 120 test (800 total)
- **caries_advanced**: 560 train / 120 val / 120 test (800 total)
- **Total**: 1,400 train / 300 val / 300 test (2,000 total)

In [ ]:
# 2. Inspect Dataset & Class Distribution per Split
labels_csv = Path("data/processed/classifier/labels.csv")
if not labels_csv.exists():
    raise FileNotFoundError(f"Labels CSV not found at {labels_csv}. Run data preparation first.")

df = pd.read_csv(labels_csv)
print(f"Total Dataset Records: {len(df):,}")

# Display contingency table of label distribution per split
split_order = ["train", "val", "test"]
class_order = ["healthy", "caries_early", "caries_advanced"]

crosstab = pd.crosstab(
    df["label"],
    df["split"],
    margins=True,
    margins_name="Total",
)

# Reorder columns and index for clean presentation
crosstab = crosstab.reindex(index=class_order + ["Total"], columns=split_order + ["Total"])

print("\nClass Distribution per Split:")
display(crosstab)

## 3. 🚀 Training (run this cell yourself)

**This cell is NOT pre-executed. Run it interactively to train the classifier and watch progress live.**

Training uses hyperparameters from `configs/classifier.yaml` (AdamW optimizer, lr=3e-4, 30 epochs with early stopping patience=7, class-weighted Cross-Entropy loss). Checkpoints and metric histories are saved to `models/classifier_runs/v0/`.

In [ ]:
# 3. Launch Classifier Training via Modular Pipeline
from dental_model.classifier.train import train

config_path = "configs/classifier.yaml"
print(f"Starting classifier training with config: {config_path}")
best_weights = train(config_path=config_path)
print(f"\nTraining complete! Best checkpoint saved to: {best_weights}")

## 4. 📈 Training & Validation Curves

Load `models/classifier_runs/v0/results.csv` and visualize:
1. **Loss Curves**: Training vs. Validation Loss across epochs.
2. **Metric Progression**: Training Accuracy, Validation Accuracy, and Validation Macro F1 score.

In [ ]:
# 4. Load Results CSV & Plot Training / Validation Curves
results_csv = Path("models/classifier_runs/v0/results.csv")

if results_csv.exists():
    history_df = pd.read_csv(results_csv)
    print(f"Total Epochs Completed: {len(history_df)}")
    display(history_df)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Subplot 1: Loss Curves
    ax1.plot(
        history_df["epoch"],
        history_df["train_loss"],
        label="Train Loss",
        marker="o",
        color="#1f77b4",
    )
    ax1.plot(
        history_df["epoch"],
        history_df["val_loss"],
        label="Val Loss",
        marker="s",
        color="#ff7f0e",
    )
    ax1.set_title("Cross-Entropy Loss vs. Epoch", fontsize=13, fontweight="bold")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend()

    # Subplot 2: Accuracy & Macro F1 Curves
    ax2.plot(
        history_df["epoch"],
        history_df["train_acc"],
        label="Train Acc",
        marker="o",
        color="#2ca02c",
    )
    ax2.plot(
        history_df["epoch"],
        history_df["val_acc"],
        label="Val Acc",
        marker="s",
        color="#17becf",
    )
    ax2.plot(
        history_df["epoch"],
        history_df["val_f1"],
        label="Val Macro F1",
        marker="^",
        color="#d62728",
    )
    ax2.set_title("Accuracy & Macro F1 vs. Epoch", fontsize=13, fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score")
    ax2.set_ylim(0, 1.05)
    ax2.grid(True, linestyle="--", alpha=0.6)
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print(f"results.csv not found at {results_csv}. Please run the training cell above first.")

## 5. 🎯 Test Split Evaluation: Confusion Matrix & Classification Report

Evaluate the best model checkpoint (`models/classifier_runs/v0/best.pt`) on the held-out test split (300 images).

In [ ]:
# 5. Evaluate Best Checkpoint on Held-out Test Split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

from dental_model.classifier.model import DentalClassifier
from dental_model.classifier.train import (
    DentalClassificationDataset,
    get_transforms,
    load_config,
)

cfg = load_config("configs/classifier.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classes = cfg["data"]["classes"]
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(classes)}

best_ckpt_path = Path("models/classifier_runs/v0/best.pt")
if not best_ckpt_path.exists():
    print(f"Checkpoint not found at {best_ckpt_path}. Please train the model first.")
else:
    ckpt = torch.load(best_ckpt_path, map_location=device)
    model = DentalClassifier.from_config(cfg).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    # Build Test DataLoader
    test_df = df[df["split"] == "test"].reset_index(drop=True)
    imgsz = cfg["data"].get("image_size", 224)
    eval_transform = get_transforms(imgsz=imgsz, is_train=False)
    test_dataset = DentalClassificationDataset(test_df, class_to_idx, transform=eval_transform)
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg["train"].get("batch_size", 32),
        shuffle=False,
        num_workers=2 if os.name != "nt" else 0,
    )

    all_preds = []
    all_targets = []
    all_probs = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1).cpu().numpy()
            preds = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_targets.extend(targets.numpy())
            all_probs.extend(probs)

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    all_probs = np.array(all_probs)

    # 1. Per-Class Precision, Recall, F1
    print("=" * 65)
    print("           TEST SPLIT CLASSIFICATION REPORT")
    print("=" * 65)
    print(classification_report(all_targets, all_preds, target_names=classes, digits=4))

    # 2. Confusion Matrix Display
    cm = confusion_matrix(all_targets, all_preds)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap="Blues", values_format="d", ax=ax, colorbar=True)
    plt.title(
        "Dental Lesion Classifier — Test Confusion Matrix",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    plt.grid(False)
    plt.tight_layout()
    plt.show()

## 6. 🔍 Sample Predictions & Misclassification Error Analysis

Review sample test images with true vs. predicted labels:
- **Misclassified Examples**: Specifically inspected to understand false-positive / false-negative boundaries (e.g. early vs. advanced enamel caries).
- **Correct Predictions**: Demonstrating high-confidence classifications across stages.

In [ ]:
# 6. Visual Inspection: Misclassified & Correctly Predicted Samples
if "all_preds" in locals():
    misclassified_indices = np.where(all_preds != all_targets)[0]
    correct_indices = np.where(all_preds == all_targets)[0]

    print(f"Total Test Images     : {len(test_df)}")
    acc_pct = len(correct_indices) / len(test_df) * 100
    print(f"Correct Predictions   : {len(correct_indices)} ({acc_pct:.2f}%)")
    err_pct = len(misclassified_indices) / len(test_df) * 100
    print(f"Misclassified Images  : {len(misclassified_indices)} ({err_pct:.2f}%)")

    def _load_rgb_image(row):
        raw_p = Path(row["image_path"])
        if raw_p.exists():
            p = raw_p
        else:
            p = REPO_ROOT / "data" / str(row.get("relative_path", ""))
        img = cv2.imread(str(p))
        if img is None:
            return np.zeros((224, 224, 3), dtype=np.uint8)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # 1. Misclassified Samples Grid (Crucial for clinical error review)
    if len(misclassified_indices) > 0:
        n_show = min(8, len(misclassified_indices))
        sample_mis = misclassified_indices[:n_show]

        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()

        for i, idx in enumerate(sample_mis):
            row = test_df.iloc[idx]
            img = _load_rgb_image(row)
            true_lbl = idx_to_class[all_targets[idx]]
            pred_lbl = idx_to_class[all_preds[idx]]
            conf = all_probs[idx][all_preds[idx]] * 100

            axes[i].imshow(img)
            axes[i].set_title(
                f"True: {true_lbl}\nPred: {pred_lbl} ({conf:.1f}%)",
                color="darkred",
                fontsize=11,
                fontweight="bold",
            )
            axes[i].axis("off")

        for j in range(len(sample_mis), len(axes)):
            axes[j].axis("off")

        plt.suptitle(
            "🚨 Sample Misclassified Test Images (Error Analysis)",
            fontsize=14,
            fontweight="bold",
            y=0.98,
        )
        plt.tight_layout()
        plt.show()
    else:
        print("No misclassifications on the test split!")

    # 2. Correctly Classified Samples Grid
    if len(correct_indices) > 0:
        n_show_correct = min(8, len(correct_indices))
        sample_cor = correct_indices[:n_show_correct]

        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()

        for i, idx in enumerate(sample_cor):
            row = test_df.iloc[idx]
            img = _load_rgb_image(row)
            true_lbl = idx_to_class[all_targets[idx]]
            pred_lbl = idx_to_class[all_preds[idx]]
            conf = all_probs[idx][all_preds[idx]] * 100

            axes[i].imshow(img)
            axes[i].set_title(
                f"True: {true_lbl}\nPred: {pred_lbl} ({conf:.1f}%)",
                color="darkgreen",
                fontsize=11,
                fontweight="bold",
            )
            axes[i].axis("off")

        for j in range(len(sample_cor), len(axes)):
            axes[j].axis("off")

        plt.suptitle(
            "✅ Sample Correctly Classified Test Images",
            fontsize=14,
            fontweight="bold",
            y=0.98,
        )
        plt.tight_layout()
        plt.show()
else:
    print("Please run the test evaluation cell above first.")